# TP - Étude de cas – Analyse Big Data de données d'accidents de la route à l'aide de PySpark

In [2]:
import sys
import importlib.util
import numpy as np
import csv
import jupyter
import pyspark
import os
from pathlib import Path
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from pyspark.sql.types import LongType
from pyspark.sql.types import FloatType
from pyspark.rdd import RDD
from pyspark.sql.types import StringType
from pyspark.sql.functions import col
import pyspark.sql.functions as F
import csv
from datetime import datetime
from functools import reduce
import pandas as pd
import matplotlib.pyplot as plt

## Configuration de l'environnement

In [3]:
# Le code est exécuté localement et utilise tous les threads disponibles.
master = "local[*]"

# Définition du nom affiché sur la page UI du cluster Spark
app_name = "Big data Analysis of Road Crash Data"

# Définition de la configuration des paramètres pour Spark
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

# Création de la session Spark
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

26/10/16 05:14:50 WARN Utils: Your hostname, felipe-Aspire-A515-57 resolves to a loopback address: 127.0.1.1; using 10.188.71.18 instead (on interface wlp42s0)
26/10/16 05:14:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 05:14:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## RDDs

Un RDD (Resilient Distributed Dataset) constitue l'abstraction fondamentale de Spark pour le traitement des données. Il s'agit, en somme, d'une collection distribuée et immuable d'objets. Spark répartit automatiquement les données contenues dans les RDD au sein du cluster et parallélise les opérations qui y sont appliquées. Chaque RDD est divisé en plusieurs partitions, lesquelles peuvent être traitées sur différents nœuds ou cœurs du cluster. Dans Spark, tout travail se traduit soit par la création de nouveaux RDD, soit par la transformation de RDD existants, soit par l'exécution d'opérations sur des RDD afin d'obtenir un résultat.

In [4]:
# Importation et création des RDD
units_path = os.path.abspath("./units_data/*.csv")
crash_path = os.path.abspath("./crash_data/*.csv")

units_rdd = sc.textFile("file://" + units_path)
crash_rdd = sc.textFile("file://" + crash_path)

# Suppression de le header des RDD
units_header = units_rdd.first()
units_rdd = units_rdd.filter(lambda row: row != units_header)
print("Nombre total d'unités_rdd :", units_rdd.count())
print(units_rdd.take(10))

crash_header = crash_rdd.first()
crash_rdd = crash_rdd.filter(lambda row: row != crash_header)
print("Nombre total de crash_rdd :", crash_rdd.count())
print(crash_rdd.take(10))


print("\nDonnées sur les unités :")
print(f"Total des partitions: {units_rdd.getNumPartitions()}")
print("Données sur les accidents :")
print(f"Total des partitions: {crash_rdd.getNumPartitions()}")

Nombre total d'unités_rdd : 272591
['"2012-1-21/08/2019","01",0,,"Pedal Cycle",,"South West","Male","035",,,,,"Straight Ahead","001","5074",,', '"2012-1-21/08/2019","02",0,"SA","Motor Cars - Sedan","2009","North West","Female","050","SA","C ","Full","Unknown","Leaving Private Driveway","001","5089",,', '"2012-2-21/08/2019","01",0,"SA","Motor Cars - Sedan","1999","West","Male","027","SA",,"Provisional 1 ","Not Towing","Straight Ahead","002","5008",,', '"2012-2-21/08/2019","02",0,"SA","Motor Cars - Sedan","1993","West",,,,,,"Not Towing","Parked","000",,,', '"2012-3-21/08/2019","01",0,"SA","Motor Cars - Sedan","XXXX","North East","Male","050","SA",,"Full","Trailer","Straight Ahead","006","5107",,', '"2012-3-21/08/2019","02",0,"SA","Motor Cars - Sedan","2006","North East","Male","044","SA",,"Full","Not Towing","Straight Ahead","003","5092",,', '"2012-3-21/08/2019","03",0,,"Other Inanimate Object",,,,,,,,,,,,,', '"2012-4-21/08/2019","01",0,"SA","Station Wagon","2009","South West","Female","

### Partitionnement des véhicules selon leur état d’immatriculation

In [5]:
# Création d'un RDD clé-valeur avec l'état d'immatriculation comme clé
def parseRecord(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))
    return (array_line[3], [x for i, x in enumerate(array_line) if i != 3])

units_state_rdd = units_rdd.map(parseRecord)

# Répartition des données en deux partitions selon l'état d'immatriculation
def hash_SA(key):
    if key == 'SA':
        return 0
    else:
        return 1

SA_partitioned_rdd = units_state_rdd.partitionBy(2, hash_SA)

# Affichage du nombre d'enregistrements dans chaque partition
def print_partitions(data):
    if isinstance(data, RDD):
        numPartitions = data.getNumPartitions()
        partitions = data.glom().collect()
    else:
        numPartitions = data.rdd.getNumPartitions()
        partitions = data.rdd.glom().collect()

    print(f"NUMBER OF PARTITIONS: {numPartitions}")
    for index, partition in enumerate(partitions):
        if len(partition) > 0:
            print(f"Partition {index}: {len(partition)} records")

print_partitions(SA_partitioned_rdd)

NUMBER OF PARTITIONS: 2
Partition 0: 218366 records
Partition 1: 54225 records


### Calcul de l'âge moyen des conducteurs selon le sexe

In [6]:
# Extraction du sexe et de l'âge des conducteurs
def parseGenderAge(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))

    # Exclusion des âges inconnus et des sexes non renseignés
    if (array_line[8] != 'XXX') & (array_line[8] != '') & (array_line[7] != 'Unknown'):
        return (array_line[7], int(array_line[8]))

gender_age_rdd = units_rdd.map(parseGenderAge)
filtered_gender_age_rdd = gender_age_rdd.filter(lambda x: x is not None).filter(lambda x: x != "")

# Regroupement par sexe et calcul de l'âge moyen
gender_age_avg_rdd = filtered_gender_age_rdd.groupByKey().mapValues(lambda x: round(sum(x) / len(x), 2))

gender_age_avg_rdd.collect()

[('Male', 40.8), ('Female', 40.08)]

### Identification des cinq véhicules les plus anciens et les plus récents

In [7]:
# Extraction de l'année, de l'état d'immatriculation et du type de véhicule
def parseYear(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))

    # Exclusion des années inconnues ou non renseignées
    if (array_line[5] != 'XXXX') & (array_line[5] != ''):
        return int(array_line[5]), [array_line[3], array_line[4]]

year_rdd = units_rdd.map(parseYear)
clean_year_rdd = year_rdd.filter(lambda x: x is not None).filter(lambda x: x != "")

print("Les plus anciens:\n", clean_year_rdd.sortByKey(ascending=True).collect()[0:5])
print("\nLes plus récents:\n", clean_year_rdd.sortByKey(ascending=False).collect()[0:5])

Les plus anciens:
 [(1900, ['VIC', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle'])]

Les plus récents:
 [(2019, ['SA', 'Station Wagon']), (2019, ['SA', 'OMNIBUS']), (2019, ['SA', 'Motor Cars - Sedan']), (2019, ['SA', 'Station Wagon']), (2019, ['SA', 'SEMI TRAILER'])]


## Dataframe

Un DataFrame est une collection distribuée de données organisées en colonnes nommées. Il équivaut à une table dans une base de données relationnelle, mais il bénéficie d’optimisations plus poussées en arrière-plan.

In [8]:
# Importation des données en DataFrame pour une analyse plus approfondie
units_path = os.path.abspath("./units_data/*.csv")
crash_path = os.path.abspath("./crash_data/*.csv")

units_df = spark.read.format('csv').option('header',True).option('escape','"').load("file://" + units_path)
crash_df = spark.read.format('csv').option('header',True).option('escape','"').load("file://" + crash_path)


### Recherche d'accidents à Adélaïde où le nombre total de victimes est supérieur à 3.

In [9]:
# changement du type de la colonne "Total Cas" en entier
crash_df = crash_df.withColumn('Total Cas',F.col('Total Cas').cast(IntegerType()))
# Application d'un filtre pour sélectionner les accidents avec plus de 3 victimes à Adélaïde
adelaide_crash_casualty_df = crash_df.filter(col("Total Cas") > 3).filter(col("Suburb") == 'ADELAIDE')
adelaide_crash_casualty_df.collect()

[Row(REPORT_ID='2012-4842-21/08/2019', Stats Area='1 City', Suburb='ADELAIDE', Postcode='5000', LGA Name='CITY OF ADELAIDE', Total Units='2', Total Cas=4, Total Fats='0', Total SI='0', Total MI='4', Year='2012', Month='March', Day='Thursday', Time='04:00 pm', Area Speed='050', Position Type='Not Divided', Horizontal Align='Straight road', Vertical Align='Level', Other Feat='Not Applicable', Road Surface='Sealed', Moisture Cond='Dry', Weather Cond='Not Raining', DayNight='Daylight', Crash Type='Rear End', Unit Resp='01', Entity Code='Driver Rider', CSEF Severity='2: MI', Traffic Ctrls='No Control', DUI Involved=None, Drugs Involved=None, ACCLOC_X='1327421.54', ACCLOC_Y='1669848.73', UNIQUE_LOC='13274221669849'),
 Row(REPORT_ID='2012-5769-21/08/2019', Stats Area='1 City', Suburb='ADELAIDE', Postcode='5000', LGA Name='CITY OF ADELAIDE', Total Units='2', Total Cas=4, Total Fats='0', Total SI='0', Total MI='4', Year='2012', Month='March', Day='Friday', Time='09:30 am', Area Speed='050', Pos

### Affichage des 10 accidents ayant entraîné le plus grand nombre de victimes.

In [10]:
crash_df.sort(col('Total Cas'), ascending=False).show(10)

+--------------------+--------------+-------------+--------+--------------------+-----------+---------+----------+--------+--------+----+--------+---------+--------+----------+-------------+--------------------+--------------+--------------+------------+-------------+------------+--------+-----------+---------+------------+-------------+---------------+------------+--------------+----------+----------+--------------+
|           REPORT_ID|    Stats Area|       Suburb|Postcode|            LGA Name|Total Units|Total Cas|Total Fats|Total SI|Total MI|Year|   Month|      Day|    Time|Area Speed|Position Type|    Horizontal Align|Vertical Align|    Other Feat|Road Surface|Moisture Cond|Weather Cond|DayNight| Crash Type|Unit Resp| Entity Code|CSEF Severity|  Traffic Ctrls|DUI Involved|Drugs Involved|  ACCLOC_X|  ACCLOC_Y|    UNIQUE_LOC|
+--------------------+--------------+-------------+--------+--------------------+-----------+---------+----------+--------+--------+----+--------+---------+--

### Nombre total de décès constatés pour chaque type d’accident.

In [11]:
crash_df.groupBy('Crash Type').agg(F.sum('Total Fats').alias('Total fatalities')).sort(col('Total fatalities'), ascending=False).show()

+--------------------+----------------+
|          Crash Type|Total fatalities|
+--------------------+----------------+
|    Hit Fixed Object|           245.0|
|             Head On|           136.0|
|      Hit Pedestrian|           109.0|
|           Roll Over|            93.0|
|         Right Angle|            82.0|
|            Rear End|            33.0|
|          Side Swipe|            30.0|
|          Right Turn|            26.0|
|  Hit Parked Vehicle|            10.0|
|          Hit Animal|             7.0|
|               Other|             5.0|
|Left Road - Out o...|             3.0|
|  Hit Object on Road|             2.0|
+--------------------+----------------+



### Affichage du nom de la banlieue et du nombre total de victimes pour chaque banlieue, dans les cas où le véhicule était conduit par une personne sans permis.

In [12]:
complete_df = crash_df.join(units_df,crash_df.REPORT_ID==units_df.REPORT_ID,how='inner')
by_suburb_no_lic_no_df = complete_df.filter(col('Licence Type') == 'Unlicenced').groupby('Suburb').agg(F.sum('Total Cas').alias('Total casualty'))
by_suburb_no_lic_no_df.sort(col('Total casualty'), ascending=False).show()


+---------------+--------------+
|         Suburb|Total casualty|
+---------------+--------------+
|       ADELAIDE|            43|
|      SALISBURY|            30|
|       PROSPECT|            27|
|     INGLE FARM|            22|
| SALISBURY EAST|            20|
|  MORPHETT VALE|            20|
|  MURRAY BRIDGE|            19|
|      DRY CREEK|            19|
| NORTH ADELAIDE|            17|
|   MAWSON LAKES|            17|
|        ENFIELD|            16|
|    GEPPS CROSS|            15|
|   REGENCY PARK|            15|
|SALISBURY DOWNS|            15|
|         SEATON|            14|
|PARA HILLS WEST|            14|
| ELIZABETH PARK|            14|
|  MOUNT GAMBIER|            13|
|   BEDFORD PARK|            13|
|   DAVOREN PARK|            13|
+---------------+--------------+
only showing top 20 rows



La gravité de l’accident est indiquée par la colonne « CSEF Severity ». De même, les colonnes « DUI Involved » et « Drugs Involved » précisent si, respectivement, la présence d’alcool ou de drogues a été détectée chez le conducteur.
Grâce à ces informations contenues dans le jeu de données, on peut analyser si la gravité des accidents est plus élevée lorsque le conducteur est sous l’emprise de drogues ou d’alcool, par rapport à une situation où il ne l’est pas.

### Le nombre total d’accidents pour chaque niveau de gravité

In [13]:
severe_df = crash_df.groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('Total accidents'))
severe_df.show()

+-------------+---------------+
|CSEF Severity|Total accidents|
+-------------+---------------+
|     4: Fatal|            722|
|        2: MI|          37300|
|       1: PDO|          84775|
|        3: SI|           4875|
+-------------+---------------+



### Le nombre total d’accidents pour chaque niveau de gravité ainsi que le pourcentage correspondant aux quatre scénarios différents

#### Lorsque le conducteur est testé positif aux drogues

In [14]:
severe_on_drugs_df = crash_df.filter(col('Drugs Involved') == 'Y').groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('Total accidents'))
total_cas_on_drugs = severe_on_drugs_df.groupBy().sum().collect()[0][0]

def find_severe_percent(s):
    return str((round((int(s)/total_cas_on_drugs)*100, 2)))+"%"

find_severe_percent_udf = udf(find_severe_percent, StringType())
severe_on_drugs_percent_df = severe_on_drugs_df.withColumn('percent_share',find_severe_percent_udf('Total accidents'))
severe_on_drugs_percent_df.show()

+-------------+---------------+-------------+
|CSEF Severity|Total accidents|percent_share|
+-------------+---------------+-------------+
|     4: Fatal|            117|        6.31%|
|        2: MI|           1109|       59.82%|
|       1: PDO|            237|       12.78%|
|        3: SI|            391|       21.09%|
+-------------+---------------+-------------+



#### Lorsque le conducteur est contrôlé avec un taux d’alcoolémie positif

In [ ]:
severe_on_alcohol_df = crash_df.filter(col('DUI Involved') == 'Y').groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('Total accidents'))
total_cas_on_alcohol = severe_on_alcohol_df.groupBy().sum().collect()[0][0]

def find_severe_percent(s):
    return str((round((int(s)/total_cas_on_alcohol)*100, 2)))+"%"

find_severe_percent_udf = udf(find_severe_percent, StringType())
severe_on_alcohol_percent_df = severe_on_alcohol_df.withColumn('percent_share',find_severe_percent_udf('Total accidents'))
severe_on_alcohol_percent_df.show()

+-------------+---------------+-------------+
|CSEF Severity|Total accidents|percent_share|
+-------------+---------------+-------------+
|     4: Fatal|            136|        3.42%|
|        2: MI|           1243|       31.24%|
|       1: PDO|           2144|       53.88%|
|        3: SI|            456|       11.46%|
+-------------+---------------+-------------+



#### Lorsque le conducteur est testé positif à la fois aux stupéfiants et à l’alcoolémie

In [ ]:
severe_on_alcohol_drugs_df = crash_df.filter((col('DUI Involved') == 'Y') & (col('Drugs Involved') == 'Y')).groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('Total accidents'))
total_cas_on_alcohol_drugs = severe_on_alcohol_drugs_df.groupBy().sum().collect()[0][0]

def find_severe_percent(s):
    return str((round((int(s)/total_cas_on_alcohol_drugs)*100, 2)))+"%"
    
find_severe_percent_udf = udf(find_severe_percent, StringType())
severe_on_alcohol_drugs_percent_df = severe_on_alcohol_drugs_df.withColumn('percent_share',find_severe_percent_udf('Total accidents'))
severe_on_alcohol_drugs_percent_df.show()

+-------------+---------------+-------------+
|CSEF Severity|Total accidents|percent_share|
+-------------+---------------+-------------+
|        2: MI|            151|       50.67%|
|       1: PDO|             36|       12.08%|
|        3: SI|             72|       24.16%|
|     4: Fatal|             39|       13.09%|
+-------------+---------------+-------------+



#### Lorsque le conducteur est testé négatif aux deux (absence d’alcool et de drogues)

In [17]:
without_drugs_alcohol_df = crash_df.filter((col('DUI Involved').isNull()) & (col('Drugs Involved').isNull()))
severe_without_drugs_alcohol_df = without_drugs_alcohol_df.groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('Total accidents'))
total_cas_without_alcohol_drugs = severe_without_drugs_alcohol_df.groupBy().sum().collect()[0][0]
def find_severe_percent(s):
    return str((round((int(s)/total_cas_without_alcohol_drugs)*100, 2)))+"%"
    
find_severe_percent_udf = udf(find_severe_percent, StringType())
severe_without_alcohol_drugs_percent_df = severe_without_drugs_alcohol_df.withColumn('percent_share',find_severe_percent_udf('Total accidents'))
severe_without_alcohol_drugs_percent_df.show()

+-------------+---------------+-------------+
|CSEF Severity|Total accidents|percent_share|
+-------------+---------------+-------------+
|     4: Fatal|            508|        0.42%|
|        2: MI|          35099|       28.74%|
|       1: PDO|          82430|       67.49%|
|        3: SI|           4100|        3.36%|
+-------------+---------------+-------------+



#### Version combinée des 4 derniers cas

In [20]:
drugs_positive_df = crash_df.filter(col('Drugs Involved') == 'Y').groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Drugs'))
alcohol_positive_df = crash_df.filter(col('DUI Involved') == 'Y').groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Alcohol'))
alcohol_drugs_positive_df = crash_df.filter((col('DUI Involved') == 'Y') & (col('Drugs Involved') == 'Y')).groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Both'))
without_drugs_alcohol_df = crash_df.filter((col('DUI Involved').isNull()) & (col('Drugs Involved').isNull()))
drugs_alcohol_negative_df = without_drugs_alcohol_df.groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On None'))

all_df = drugs_positive_df.join(alcohol_positive_df, ["CSEF Severity"]).join(alcohol_drugs_positive_df, ["CSEF Severity"]).join(drugs_alcohol_negative_df, ["CSEF Severity"])
all_df.show()

total_cas_on_drugs = drugs_positive_df.groupBy().sum().collect()[0][0]

+-------------+--------+----------+-------+-------+
|CSEF Severity|On Drugs|On Alcohol|On Both|On None|
+-------------+--------+----------+-------+-------+
|     4: Fatal|     117|       136|     39|    508|
|        2: MI|    1109|      1243|    151|  35099|
|       1: PDO|     237|      2144|     36|  82430|
|        3: SI|     391|       456|     72|   4100|
+-------------+--------+----------+-------+-------+

